# 风电场布局优化问题 (WFLOP)

**类别:** 选址

来源: [https://www.hexaly.com/templates/wind-farm-layout-optimization-problem-wflop](https://www.hexaly.com/templates/wind-farm-layout-optimization-problem-wflop)

## 问题

**风电场布局优化问题 (WFLOP)** 涉及在指定区域内确定固定数量的风力涡轮机的最佳布置位置。每台涡轮机必须位于该区域内部,并且与其他涡轮机之间保持最小安全距离。目标是最大化年发电量 (AEP),这是衡量风电场将风能转化为电能效率的关键指标。该优化问题的核心挑战在于最小化尾流损失,即涡轮机之间相互干扰其风流所产生的损失。

虽然许多方法处理该问题的连续版本,但我们的工作聚焦于一种离散近似,从而能够在各种可行域类型上进优化。在本示例中,我们关注圆形区域,但该方法同样适用于任何几何形状。

### 学到的建模技巧

- 使用 [JSON 模块](https://www.hexaly.com/docs/last/modelerreference/standardlibrary/jsonmodule.html)读取输入文件
- 使用[非线性运算符](https://www.hexaly.com/docs/last/modelingfeatures/mathematicalmodelingfeatures.html#table-of-available-operators-and-functions)计算尾流损失和功率
- 最大化一个[非线性目标函数](https://www.hexaly.com/docs/last/guidelines/modelingprinciples.html#define-your-objective-function) (AEP)

## 数据

我们使用 [IEA Task 37 on System Engineering in Wind Energy](https://github.com/byuflowlab/iea37-wflo-casestudies/tree/master/cs1-2) 提供的实例。其中包含三种场景,分别涉及 16、36 和 64 台风力涡轮机,部署在半径分别为 1300 m、2000 m 和 3000 m 的圆形区域内。我们已将原始的 .yaml 数据文件转换为 .json 格式以便处理。

每个 .json 文件包含以下参数:

- "radius": 部署区域半径
- "D": 风轮直径
- "nT": 涡轮机数量
- "prated": 单台涡轮机的额定功率
- "urated": 额定风速
- "ucut-in": 切入风速
- "ucut-out": 切出风速
- "wind": 风数据,包括:

- "degrees": 风向角度的离散化
- "probs": 风向的概率分布
- "J": 考虑的风向总数
- "speed": 来流风速
- "cT": 推力系数
- "kY": 基于湍流强度 0.075 的动态系数

可行区域以间距 1.7 × D 进行均匀离散化,涡轮机之间的最小距离为 2 × D。涡轮机既可放置在圆形区域内的离散网格点上,也可放置在边界上,其中针对每个角度评估一个可行位置(详见下文)。

## 模型

用于风电场布局优化问题 (WFLOP) 的 Hexaly 模型使用一个二元决策向量,其中每个元素表示对应离散位置上是否放置风力涡轮机(1 表示放置,0 表示不放置)。

第一个约束确保可行区域内放置的涡轮机总数等于 nT。

为强制最小距离要求,我们对任意一对距离小于允许阈值的位置施加约束,使得这两个位置中至多只能放置一台涡轮机。

每个潜在位置的风条件采用 [Bastankhah 高斯尾流模型的简化版本](https://github.com/byuflowlab/iea37-wflo-casestudies/blob/master/cs1-2/iea37-wakemodel.pdf) 进行计算,涡轮机的功率输出采用一条平滑、凸且单调递增的简化功率曲线得到(参见 [IEA Task 37 Anouncements p2](https://github.com/byuflowlab/iea37-wflo-casestudies/blob/master/cs1-2/iea37-wflocs-announcement.pdf#page=2))。

目标函数通过聚合所有风向下的期望功率输出(以各自概率加权)来最大化[年发电量 (AEP)](https://github.com/byuflowlab/iea37-wflo-casestudies/blob/master/cs1-2/iea37-wflocs-announcement.pdf#page=1)。

## Python 实现

In [ ]:
# Copyright (c) Hexaly. Permission is hereby granted to use, copy,
# and modify this code for applications developed with Hexaly.
import hexaly.optimizer
import json
import math
import sys

def main(instance_file, output_file, time_limit):
    """
    Creates and solves the model.
    """
    # Read the input data
    degrees, nb_degrees, probabilities, inflow_speed, thrust_coeff,  dyn_coeff, \
            radius, nb_turbines, turbine_diameter, turbine_nominal_power, min_distance, \
            optimal_speed, min_speed, max_speed, disc_delta = read_data(instance_file)
    ## Additional required inputs
    points_x, points_y = build_discretization(radius, disc_delta)
    nb_locations = len(points_x)
    incompatibility_sets = compute_incompatibilities(points_x, points_y, min_distance)

    with hexaly.optimizer.HexalyOptimizer() as optimizer:
        # Declare the optimization model
        model = optimizer.model

        # Decision Variable : List of booleans (one for each Wind Turbine)
        chosen_turbines = [model.bool() for location in range(nb_locations)]

        # Constraint : We have to choose nT locations on the nb_locations available positions
        model.constraint(sum(chosen_turbines[location] for location in range(nb_locations)) == nb_turbines)
        # Constraint : Two WT need to be spaced by at least d_min meters
        for location in range(nb_locations):
            for incompatible_location in incompatibility_sets[location]:
                model.constraint(chosen_turbines[location] + chosen_turbines[incompatible_location] <= 1)

        # Relative wind and power at each location for every possible wind angle
        turbine_wind = [[0 for angle in range(nb_degrees)] for location in range(nb_locations)]
        turbine_power = [[0 for angle in range(nb_degrees)] for location in range(nb_locations)]
        for location_1 in range(nb_locations):
            wt_1 = [points_x[location_1], points_y[location_1]]
            for angle in range(nb_degrees):
                # Wind at location i for a wind incidence angle j
                w_loss_sum = sum(chosen_turbines[location_2]
                        * (wake_loss(wt_1, [points_x[location_2], points_y[location_2]],
                                degrees[angle], dyn_coeff, turbine_diameter, thrust_coeff))**2
                                for location_2 in range(nb_locations))
                turbine_wind[location_1][angle] =  inflow_speed * (chosen_turbines[location_1]
                        - model.sqrt(w_loss_sum))

                # Power of the wind turbine of a WT at i with a wind incidence angle j
                polynomial_case = (min_speed <= turbine_wind[location_1][angle]) \
                        * (turbine_wind[location_1][angle] < optimal_speed)
                polynomial_value = ((turbine_wind[location_1][angle] - min_speed) \
                        / (optimal_speed - min_speed))**3
                constant_case = (optimal_speed <= turbine_wind[location_1][angle]) \
                        * (turbine_wind[location_1][angle] < max_speed)
                turbine_power[location_1][angle] = polynomial_case \
                        * turbine_nominal_power * polynomial_value \
                        +  constant_case * turbine_nominal_power


        # Objective : we maximize the Annual Energy Production (in Wh)
        # To calculate the AEP, we multiply the total power produced by the number of hours in one year
        objective = 8760 * sum(probabilities[angle] * turbine_power[location][angle]
                for location in range(nb_locations) for angle in range(nb_degrees))
        model.maximize(objective)

        # Finalize model and solve
        model.close()
        optimizer.param.time_limit = time_limit
        optimizer.solve()

        feasible_list = ["HxSolutionStatus.FEASIBLE", "HxSolutionStatus.OPTIMAL"]
        if (str(optimizer.solution.status) in feasible_list):
            # Store the solution in a map
            points = []
            for location in range(nb_locations):
                if (chosen_turbines[location].value == 1):
                    points.append((points_x[location], points_y[location]))

            # If asked, we print the result in the output file
            if (output_file is not None):
                with open(output_file, "w") as f:
                    print("Solution written in file ", output_file)
                    # Print every important info of the resolution
                    f.write("*************************************\n")
                    f.write("*** WIND FARM LAYOUT OPTIMIZATION ***\n")
                    f.write("*************************************\n\n")

                    f.write("Annual Energy Production : " + str(objective.value)
                            + " Wh (" + str(round(objective.value / 1e9 * 100) / 100)
                            + " GWh).\n\n")
                    f.write("Locations of the Wind Turbines : \n")

                    for turbine in range(nb_turbines):
                        point = points[turbine]
                        f.write("   - (" + str(point[0]) + ", " + str(point[1]) + ")\n")

        else:
            print("No feasible solution have been found within the allowed time.")

def read_data(instance_file):
    """
    Reads the input files of the problem.
    """
    # Opening the .json file
    with open(instance_file, 'r') as file:
        data = json.load(file)

    # Wind information
    degrees = data["wind"]["degrees"]     # Discretization of the degrees rose
    nb_degrees = len(degrees)             # Number of discretizations
    probabilities = data["wind"]["probs"] # Probabilities of wind in every direction
    inflow_speed = data["wind"]["speed"]  # Inflow wind speed
    thrust_coeff = data["wind"]["cT"]     # Thrust coefficient 
    dyn_coeff = data["wind"]["kY"]        # Dynamic coefficient

    # Information about the problem
    radius = data["radius"]             # Radius of the perimeter to fullfil
    nb_turbines = data["nT"]            # Number of Wind Turbines
    turbine_diameter = data["D"]                   # Diameter of each Turbine
    turbine_nominal_power = data["p_rated"] * 1e6  # Power of the Turbine
    min_distance = 2 * turbine_diameter            # Minimal distance between two turbines
    optimal_speed = data["u_rated"]     # Optimal wind speed (maximum efficiency)
    min_speed = data["u_cut_in"]        # Minimal wind speed producing energy
    max_speed = data["u_cut_out"]       # Maximal wind speed producing energy

    disc_delta = 1.7 * turbine_diameter # Distance between two points in the discretization

    return degrees, nb_degrees, probabilities, inflow_speed, thrust_coeff, \
            dyn_coeff, radius, nb_turbines, turbine_diameter, turbine_nominal_power, \
            min_distance, optimal_speed, min_speed, max_speed, disc_delta

def build_discretization(radius, disc_delta):
    """
    Builds the discretization in a circular field of radius {radius} and with a
    distance between points of {disc_delta}.

    The discretization follows the one described in 'eawe' paper, i.e. building
    a regular uniform mesh inside the circle, and a point every degree on the
    frontier of the circle.
    """
    points_x = []
    points_y = []
    # Maximum number of points in any direction from the center
    max_onedir_points = int(radius / disc_delta + 1)
    # Interior points
    for c_x in range(max_onedir_points):
        for side_x in [-1, 1]:
            if (not(c_x == 0 and side_x == 1)):
                new_x = side_x * c_x * disc_delta
                for c_y in range(max_onedir_points):
                    for side_y in [-1, 1]:
                        new_y = side_y * c_y * disc_delta
                        if (math.sqrt(new_x**2 + new_y**2) < radius \
                                and not(c_y == 0 and side_y == 1)):
                            points_x.append(new_x)
                            points_y.append(new_y)

    # Points on the border of the circle
    for deg in range(360):
        points_x.append(radius * math.sin(deg * math.pi / 180))
        points_y.append(radius * math.cos(deg * math.pi / 180))

    return points_x, points_y

def compute_incompatibilities(points_x, points_y, min_distance):
    """
    Compute incompatibility sets for every location on the field.

    N_i[i] := {l in 0...nb_locations : l != i && ||l - i|| < min_distance}
    """
    # Number of available points
    nb_locations = len(points_x)
    incompatibilities = [[] for location in range(nb_locations)]
    # Fill the set for every location
    for location_1 in range(nb_locations):
        for location_2 in range(location_1+1, nb_locations):
            if (math.sqrt((points_x[location_1] - points_x[location_2])**2
                    + (points_y[location_1] - points_y[location_2])**2) < min_distance):
                incompatibilities[location_1].append(location_2)
                incompatibilities[location_2].append(location_1)

    return incompatibilities

def distances(location_1, location_2, theta):
    """
    This function calculates the distance between two points, considering a
    frame of reference with angle theta.
    """
    # Rotation of the reference
    theta_deg = 270 - theta
    theta_rad = theta_deg * math.pi / 180
    cos_wind = math.cos(-theta_rad)
    sin_wind = math.sin(-theta_rad)

    # Change the reference of the coordinates
    location_2_x = (location_2[0] * cos_wind) - (location_2[1] * sin_wind)
    location_2_y = (location_2[0] * sin_wind) + (location_2[1] * cos_wind)

    location_1_x = (location_1[0] * cos_wind) - (location_1[1] * sin_wind)
    location_1_y = (location_1[0] * sin_wind) + (location_1[1] * cos_wind)

    return {"parallel":location_1_x - location_2_x, "perpendicular":location_1_y - location_2_y}

def wake_loss(turbine_1, turbine_2, theta, dyn_coeff, turbine_diameter, thrust_coeff):
    """
    Wake loss evaluated at i, caused by a Wind Turbine at l, considering a wind
    of angle {theta}. 
    This loss is a simplified version of Bastankhah's Gaussian model.
    """
    # We calculate parallel and perpendicular distances
    d = distances(turbine_1, turbine_2, theta)
    # If x_i - x_l < 0, the wake loss is zero
    if (d["parallel"] > 0):
        # Standard deviation of the wake deficit
        s_y = dyn_coeff * d["parallel"] + turbine_diameter / math.sqrt(8)
        # Separation of the product in two components
        coeff = thrust_coeff / (8 * (s_y / turbine_diameter)**2)
        pdt_1 = 1 - math.sqrt(1 - coeff)
        pdt_2 = math.exp(-1/2 * (d["perpendicular"] / s_y)**2)

        return pdt_1 * pdt_2

    return 0

# Main
if __name__ == '__main__':
    if len(sys.argv) < 2:
        print("Usage: python wind_farm_layout_optimization.py inputFile "
              + "[solFile] [timeLimit]")
        sys.exit(1)
    # Get arguments
    instance_file = sys.argv[1]
    output_file = sys.argv[2] if len(sys.argv) >= 3 else None
    time_limit = int(sys.argv[3]) if len(sys.argv) >= 4 else 60
    # Create the model and solve it
    main(instance_file, output_file, time_limit)
